# Word Ladder
A transformation sequence from word beginWord to word endWord using a dictionary wordList is a sequence of words beginWord -> s1 -> s2 -> ... -> sk such that:

* Every adjacent pair of words differs by a single letter.
* Every $s_i$ for 1 <= i <= k is in wordList. Note that beginWord does not need to be in wordList.
* sk == endWord

Given two words, beginWord and endWord, and a dictionary wordList, return the number of words in the shortest transformation sequence from beginWord to endWord, or 0 if no such sequence exists.

### Example 1
Input: beginWord = "hit", endWord = "cog", wordList = ["hot","dot","dog","lot","log","cog"]\
Output: 5\
Explanation: One shortest transformation sequence is "hit" -> "hot" -> "dot" -> "dog" -> cog", which is 5 words long.

### Example 2
Input: beginWord = "hit", endWord = "cog", wordList = ["hot","dot","dog","lot","log"]\
Output: 0\
Explanation: The endWord "cog" is not in wordList, therefore there is no valid transformation sequence.

### Constraints
* 1 <= beginWord.length <= 10
* endWord.length == beginWord.length
* 1 <= wordList.length <= 5000
* wordList[i].length == beginWord.length
* beginWord, endWord, and wordList[i] consist of lowercase English letters.
* beginWord != endWord
* All the words in wordList are unique.

## Approach 1: Create a Graph and do BFS
We can create a graph `transition_map` where there are two types of nodes. The first type are all of the words from wordList. The second type are all the transitions. Transitions are strings that represent a transformation that a given word can make. For example, "dog" would have transitions ["*og", "d*g", "do*"] where the * character takes the place of the letter that's changed in the word as we create a word ladder. Words are adjacent to their transitions, and transitions are adjacent to words that fit the pattern dictated by the transition string. So, we can use the transitions to figure out which words can be connected in a word ladder.

After creating this graph, we can use BFS to find the shortest path from `beginWord` to `endWord`. In the `bfs_queue`, we'll store states in the format (`word`, `count`), and initialize the `bfs_queue` to [(beginWord, 1)]. For each state in `bfs_queue`, we'll return `count` if word == endWord. 

Otherwise, we'll continue searching for the next steps in the path. We'll generate the `transitions` for `word` and iterate over each transition to get all the `adjacent` words. Then, we iterate over `adjacent`-`seen` words and add them to the `bfs_queue` alongside an incremented `count`.

## Analysis
* Time Complexity: $O(M^{2}*N)$, M = len(beginWord), N = len(wordList)
    * generate_transitions() is $O(M^2)$ because we splice the word while also looping over each character of the word to generate the transitions
    * When we create `transition_map`, we need to create transitions for each word in wordList, which is $O(M^{2}*N)$
    * With the `bfs_queue` we iterate over each word at most once thanks to `seen` set and on each iteration we have to call generate_transitions(). So, bfs is $O(M^{2} * N)$
* Space Complexity: $O(M^{2}*N)$
    * the biggest thing is creating `transition_map` which will contain all the transitions plus their edges to corresponding words
    * For each M transition, we store a word of length M which is O(M^{2}) space stored. For all N words in wordList, that'll be $O(M^{2}*N)$ 


In [ ]:
from typing import List
from collections import deque

def generate_transitions(word):
    return {word[:i]+'*'+word[i+1:] for i in range(len(word))}

def ladderLength(beginWord: str, endWord: str, wordList: List[str]) -> int:
    if endWord not in wordList:
        return 0

    word_set = set(wordList)

    transition_map = dict()

    for word in word_set | {beginWord}:
        transitions = generate_transitions(word)
        for transition in transitions:
            if transition not in transition_map:
                transition_map[transition] = set()
            transition_map[transition].add(word)

    start = (beginWord,1)
    seen = set([beginWord])
    bfs_queue = deque([start])
    while bfs_queue:
        word, count = bfs_queue.popleft()
        if word == endWord:
            return count
        adjacent = set()
        for transition in generate_transitions(word):
            adjacent.update(transition_map[transition])
        for next_word in adjacent - seen:
            next_state = (next_word, count+1)
            bfs_queue.append(next_state)
            seen.add(next_word)
    return 0